In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
llm= ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class SubState(TypedDict):
    input_text:str
    translated_text : str

In [5]:
subgraph_llm= ChatGroq(model="llama-3.3-70b-versatile")

In [7]:
def translate_text(state:SubState):
        prompt=f""""
            translate the folowing text into hindi.
            keep it natural and real
     
                 text:
                 {state['input_text']} 
                    """.strip()
        translate_text=subgraph_llm.invoke(prompt).content

        return{'translate_text':translate_text}

In [8]:
subgraph_builder=StateGraph(SubState)

subgraph_builder.add_node('translate_text',translate_text)

subgraph_builder.add_edge(START,'translate_text')
subgraph_builder.add_edge('translate_text',END)

subgraph= subgraph_builder.compile()

In [9]:
class ParentState(TypedDict):
    question:str
    answer_eng:str
    answer_hin:str

In [10]:
parent_llm = ChatGroq(model="llama-3.3-70b-versatile")

In [11]:
def generate_answer(state: ParentState):
    answer=parent_llm.invoke(f"You are a helful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng':answer}

In [12]:
def translate_answer(state:ParentState):
    result=subgraph.invoke({'input_text':state['answer_eng']})

    return{'answer_hin':result['translate_text']}